In [4]:
"""
STEP 1: Confirmatory Factor Analysis (CFA)
==========================================
Study: Grumbling Behavior in STEM Students toward Non-STEM Courses

PURPOSE:
  Before SEM, CFA checks whether each survey question (indicator)
  correctly and reliably measures its intended construct (latent variable).
  Think of it as a quality check on your measurement instrument.

WHAT YOU WILL GET:
  1. Factor Loadings       — Does each question load strongly on its construct? (want > 0.5)
  2. Cronbach's Alpha      — Is each construct internally consistent? (want > 0.7)
  3. AVE                   — Average Variance Extracted, convergent validity (want > 0.5)
  4. CR                    — Composite Reliability (want > 0.7)
  5. Correlation Matrix    — How constructs relate to each other
  6. Discriminant Validity — Are constructs distinct from each other?

REQUIREMENTS:
  pip install pandas numpy scipy factor_analyzer semopy openpyxl
"""

import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
df = pd.read_excel('./GRUMBLING BEHAVIOR (STEM) (Responses).xlsx',sheet_name='Form Responses 1')

# ─────────────────────────────────────────────
# 2. ENCODE LIKERT SCALE (text → numbers)
# ─────────────────────────────────────────────
likert_map = {
    'Strongly disagree': 1,
    'Disagree':          2,
    'Neutral':           3,
    'Agree':             4,
    'Strongly agree':    5
}

cols = df.columns[4:19]
df_likert = df[cols].copy()
df_likert = df_likert.applymap(lambda x: likert_map.get(str(x).strip(), np.nan))
df_likert = df_likert.dropna()

print(f"✅ Valid responses after cleaning: {len(df_likert)} out of {len(df)}")

# ─────────────────────────────────────────────
# 3. DEFINE CONSTRUCTS & THEIR INDICATORS
# ─────────────────────────────────────────────
# Each construct maps to the column indices from the original survey
constructs = {
    'Switching_Cost': [
        'Adjusting my learning style for non-STEM courses is more challenging than for STEM courses. (Switching Cost)',
        'Switching focus from STEM to non-STEM subjects is mentally exhausting. (Switching Cost)'
    ],
    'Switching_Benefit': [
        'Non-STEM courses improve my soft skills, such as communication and critical thinking. (Switching Benefit)',
        'Studying non-STEM subjects broadens my overall perspective on learning.'
    ],
    'Work_Overload': [
        'Non-STEM courses significantly increase my academic workload. (Work Overload)',
        'Managing both STEM and non-STEM courses is challenging. (Work Overload)'
    ],
    'Role_Ambiguity': [
        'I am unsure how non-STEM courses contribute to my academic goals. (Role Ambiguity)',
        'I struggle to see the practical application of non-STEM subjects compared to STEM subjects. (Role Ambiguity)'
    ],
    'Work_Home_Conflict': [
        'Balancing non-STEM and STEM coursework leaves me with less time for personal activities. (Work–Home Conflict)'
    ],
    'General_Grumbling': [
        'I feel that non-STEM courses are a waste of my time compared to STEM courses. (General Grumbling)',
        'Courses in the humanities (e.g., literature, philosophy) are unnecessary for STEM students.',
        'Social science courses (e.g., sociology, psychology) seem less relevant to my field than STEM subjects."'
    ],
    'Willingness': [
        'I believe non-STEM courses are valuable despite their challenges. (Willingness)',
        'I am open to integrating lessons from non-STEM courses into my academic work.  (Willingness)'
    ],
    'Long_Term_Adoption': [
        'I will likely apply the knowledge I gain from non-STEM courses in my personal or professional life. (Long-Term Adoption)'
    ]
}

# ─────────────────────────────────────────────
# 4. HELPER FUNCTIONS
# ─────────────────────────────────────────────

def cronbach_alpha(data):
    """Calculate Cronbach's Alpha for a set of items."""
    if data.shape[1] < 2:
        return np.nan
    n = data.shape[1]
    item_vars = data.var(axis=0, ddof=1)
    total_var = data.sum(axis=1).var(ddof=1)
    return (n / (n - 1)) * (1 - item_vars.sum() / total_var)

def factor_loading(item_data, construct_data):
    """Pearson correlation of item with construct total score (item-total correlation)."""
    return item_data.corr(construct_data)

def compute_ave(loadings):
    """Average Variance Extracted = mean of squared loadings."""
    l = np.array(loadings)
    return np.mean(l**2)

def compute_cr(loadings):
    """Composite Reliability = (sum of loadings)^2 / ((sum of loadings)^2 + sum of error variances)."""
    l = np.array(loadings)
    return l.sum()**2 / (l.sum()**2 + np.sum(1 - l**2))

# ─────────────────────────────────────────────
# 5. RUN CFA — COMPUTE ALL METRICS
# ─────────────────────────────────────────────
print("\n" + "="*70)
print("        CONFIRMATORY FACTOR ANALYSIS (CFA) RESULTS")
print("="*70)

construct_scores = {}
cfa_summary = []

for construct, items in constructs.items():
    # Filter only items that exist in our dataframe
    valid_items = [i for i in items if i in df_likert.columns]

    if not valid_items:
        print(f"⚠️  No valid items found for {construct}")
        continue

    construct_data = df_likert[valid_items]
    construct_score = construct_data.mean(axis=1)
    construct_scores[construct] = construct_score

    print(f"\n{'─'*60}")
    print(f"  CONSTRUCT: {construct.replace('_', ' ')}")
    print(f"{'─'*60}")

    # Factor loadings (item-total correlation)
    loadings = []
    for item in valid_items:
        loading = factor_loading(df_likert[item], construct_score)
        flag = "✅" if loading >= 0.5 else "⚠️ LOW"
        short_name = item[:55] + "..." if len(item) > 55 else item
        print(f"  {flag}  Loading: {loading:.3f}  |  {short_name}")
        loadings.append(loading)

    # Reliability & Validity
    alpha = cronbach_alpha(construct_data) if len(valid_items) > 1 else np.nan
    ave   = compute_ave(loadings)
    cr    = compute_cr(loadings) if len(valid_items) > 1 else np.nan
    mean  = construct_score.mean()
    std   = construct_score.std()

    alpha_flag = "✅" if (np.isnan(alpha) or alpha >= 0.7) else "⚠️ LOW"
    ave_flag   = "✅" if ave >= 0.5 else "⚠️ LOW"
    cr_flag    = "✅" if (np.isnan(cr) or cr >= 0.7) else "⚠️ LOW"

    print(f"\n  Mean: {mean:.2f}  |  SD: {std:.2f}")
    print(f"  {alpha_flag} Cronbach's Alpha : {alpha:.3f}" if not np.isnan(alpha) else "  Cronbach's Alpha : N/A (single item)")
    print(f"  {ave_flag} AVE              : {ave:.3f}")
    print(f"  {cr_flag} Composite Rel.  : {cr:.3f}" if not np.isnan(cr) else "  Composite Rel.  : N/A (single item)")

    cfa_summary.append({
        'Construct':         construct.replace('_', ' '),
        'N_Items':           len(valid_items),
        'Mean':              round(mean, 3),
        'SD':                round(std, 3),
        "Cronbach's Alpha":  round(alpha, 3) if not np.isnan(alpha) else 'N/A',
        'AVE':               round(ave, 3),
        'CR':                round(cr, 3) if not np.isnan(cr) else 'N/A',
        'Min Loading':       round(min(loadings), 3),
        'Max Loading':       round(max(loadings), 3)
    })

# ─────────────────────────────────────────────
# 6. CONSTRUCT CORRELATION MATRIX
# ─────────────────────────────────────────────
print("\n\n" + "="*70)
print("        CONSTRUCT CORRELATION MATRIX")
print("="*70)
print("  (Shows how constructs relate to each other)")
print("  High correlation with Grumbling = strong driver\n")

scores_df = pd.DataFrame(construct_scores)
corr_matrix = scores_df.corr().round(3)
print(corr_matrix.to_string())

# ─────────────────────────────────────────────
# 7. DISCRIMINANT VALIDITY (AVE > r²)
# ─────────────────────────────────────────────
print("\n\n" + "="*70)
print("        DISCRIMINANT VALIDITY CHECK")
print("="*70)
print("  Rule: sqrt(AVE) of each construct should be > correlation with others\n")

ave_dict = {row['Construct'].replace(' ', '_'): row['AVE']
            for row in cfa_summary if isinstance(row['AVE'], float)}

construct_names = list(ave_dict.keys())
for i, c1 in enumerate(construct_names):
    sqrt_ave = np.sqrt(ave_dict[c1])
    for j, c2 in enumerate(construct_names):
        if j <= i:
            continue
        if c1 in scores_df.columns and c2 in scores_df.columns:
            r = scores_df[c1].corr(scores_df[c2])
            ok = "✅" if sqrt_ave > abs(r) else "⚠️  ISSUE"
            print(f"  {ok}  {c1} vs {c2}: sqrt(AVE)={sqrt_ave:.3f} > |r|={abs(r):.3f}")

# ─────────────────────────────────────────────
# 8. SUMMARY TABLE
# ─────────────────────────────────────────────
print("\n\n" + "="*70)
print("        CFA SUMMARY TABLE")
print("="*70)
summary_df = pd.DataFrame(cfa_summary)
print(summary_df.to_string(index=False))

# ─────────────────────────────────────────────
# 9. WHAT TO LOOK FOR — INTERPRETATION GUIDE
# ─────────────────────────────────────────────
print("\n\n" + "="*70)
print("        HOW TO INTERPRET THESE RESULTS")
print("="*70)
print("""
  FACTOR LOADINGS (> 0.5 is good, > 0.7 is excellent)
    → Shows how well each question represents its construct
    → Low loading = that question may need to be dropped before SEM

  CRONBACH'S ALPHA (> 0.7 is good)
    → Internal consistency — do the questions in a construct agree with each other?
    → If low, the construct questions may be measuring different things

  AVE — Average Variance Extracted (> 0.5 is good)
    → Convergent validity — the construct captures more than half the variance of its items
    → If low, items may not be strongly enough tied to the construct

  COMPOSITE RELIABILITY (> 0.7 is good)
    → More robust version of Alpha for SEM

  DISCRIMINANT VALIDITY
    → Each construct should be more related to its own items than to other constructs
    → Violations mean two constructs may be too similar to separate

  ──────────────────────────────────────────────
  ✅ If CFA results look good → proceed to STEP 2: SEM
  ⚠️  If some loadings are low → we may drop those items before SEM
  ──────────────────────────────────────────────
""")

print("\n✅ CFA COMPLETE. Share these results and we will proceed to SEM.\n")

✅ Valid responses after cleaning: 212 out of 212

        CONFIRMATORY FACTOR ANALYSIS (CFA) RESULTS

────────────────────────────────────────────────────────────
  CONSTRUCT: Switching Cost
────────────────────────────────────────────────────────────
  ✅  Loading: 0.893  |  Adjusting my learning style for non-STEM courses is mor...
  ✅  Loading: 0.876  |  Switching focus from STEM to non-STEM subjects is menta...

  Mean: 3.35  |  SD: 0.84
  ✅ Cronbach's Alpha : 0.721
  ✅ AVE              : 0.782
  ✅ Composite Rel.  : 0.878

────────────────────────────────────────────────────────────
  CONSTRUCT: Switching Benefit
────────────────────────────────────────────────────────────
  ✅  Loading: 0.883  |  Non-STEM courses improve my soft skills, such as commun...
  ✅  Loading: 0.853  |  Studying non-STEM subjects broadens my overall perspect...

  Mean: 3.47  |  SD: 0.81
  ⚠️ LOW Cronbach's Alpha : 0.672
  ✅ AVE              : 0.754
  ✅ Composite Rel.  : 0.860

──────────────────────────────

In [9]:
"""
STEP 2: Structural Equation Modeling (SEM)
==========================================
Study: Grumbling Behavior in STEM Students toward Non-STEM Courses

PURPOSE:
  Find what causes General Grumbling, how strongly each construct
  drives it, and whether any constructs work THROUGH others (mediation).

WHAT YOU WILL GET:
  1. Direct Path Coefficients   — Which constructs directly affect Grumbling
  2. Indirect Effects           — Which constructs work through other constructs
  3. Total Effects              — Full impact of each construct on Grumbling
  4. Model Fit Indices          — How well the model fits the data
  5. R²                         — How much of Grumbling is explained
  6. Ranking                    — What matters most

STRUCTURAL MODEL:
  Theory-based paths:

  Switching Cost     ──► Work Overload      ──► General Grumbling
  Switching Cost     ──► Role Ambiguity     ──► General Grumbling
  Switching Cost     ──────────────────────► General Grumbling
  Work Overload      ──► Work-Home Conflict ──► General Grumbling
  Work Overload      ──────────────────────► General Grumbling
  Role Ambiguity     ──────────────────────► General Grumbling
  Work-Home Conflict ──────────────────────► General Grumbling
  Switching Benefit  ──► Willingness        ──► General Grumbling
  Switching Benefit  ──────────────────────► General Grumbling
  Willingness        ──► Long-Term Adoption
  Willingness        ──────────────────────► General Grumbling
  Long-Term Adoption ──────────────────────► General Grumbling

REQUIREMENTS:
  pip install pandas numpy scipy semopy openpyxl
"""

import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 1. LOAD & ENCODE DATA
# ─────────────────────────────────────────────
df = pd.read_excel('./GRUMBLING BEHAVIOR (STEM) (Responses).xlsx',
                   sheet_name='Form Responses 1')

likert_map = {
    'Strongly disagree': 1,
    'Disagree':          2,
    'Neutral':           3,
    'Agree':             4,
    'Strongly agree':    5
}

cols = df.columns[4:19]
df_likert = df[cols].copy()
df_likert = df_likert.applymap(lambda x: likert_map.get(str(x).strip(), np.nan))
df_likert = df_likert.dropna()

print(f"✅ Valid responses: {len(df_likert)}")

# ─────────────────────────────────────────────
# 2. BUILD CONSTRUCT SCORES
# ─────────────────────────────────────────────
constructs = {
    'Switching_Cost': [
        'Adjusting my learning style for non-STEM courses is more challenging than for STEM courses. (Switching Cost)',
        'Switching focus from STEM to non-STEM subjects is mentally exhausting. (Switching Cost)'
    ],
    'Switching_Benefit': [
        'Non-STEM courses improve my soft skills, such as communication and critical thinking. (Switching Benefit)',
        'Studying non-STEM subjects broadens my overall perspective on learning.'
    ],
    'Work_Overload': [
        'Non-STEM courses significantly increase my academic workload. (Work Overload)',
        'Managing both STEM and non-STEM courses is challenging. (Work Overload)'
    ],
    'Role_Ambiguity': [
        'I am unsure how non-STEM courses contribute to my academic goals. (Role Ambiguity)',
        'I struggle to see the practical application of non-STEM subjects compared to STEM subjects. (Role Ambiguity)'
    ],
    'Work_Home_Conflict': [
        'Balancing non-STEM and STEM coursework leaves me with less time for personal activities. (Work–Home Conflict)'
    ],
    'General_Grumbling': [
        'I feel that non-STEM courses are a waste of my time compared to STEM courses. (General Grumbling)',
        'Courses in the humanities (e.g., literature, philosophy) are unnecessary for STEM students.',
        'Social science courses (e.g., sociology, psychology) seem less relevant to my field than STEM subjects."'
    ],
    'Willingness': [
        'I believe non-STEM courses are valuable despite their challenges. (Willingness)',
        'I am open to integrating lessons from non-STEM courses into my academic work.  (Willingness)'
    ],
    'Long_Term_Adoption': [
        'I will likely apply the knowledge I gain from non-STEM courses in my personal or professional life. (Long-Term Adoption)'
    ]
}

scores = {}
for name, items in constructs.items():
    valid = [i for i in items if i in df_likert.columns]
    scores[name] = df_likert[valid].mean(axis=1)

data = pd.DataFrame(scores)

# Standardize all construct scores (mean=0, sd=1)
# This makes path coefficients directly comparable (like standardized betas)
data_std = (data - data.mean()) / data.std()

# ─────────────────────────────────────────────
# 3. SEM VIA PATH ANALYSIS (OLS REGRESSION)
# ─────────────────────────────────────────────
# We estimate each structural equation separately.
# This is the "path analysis" approach — equivalent to SEM
# when constructs are observed (composite scores).
#
# PATH STRUCTURE (theory-driven):
#
#  Exogenous (independent, no incoming paths):
#    - Switching_Cost
#    - Switching_Benefit
#
#  Mediators (middle layer):
#    - Work_Overload       ← Switching_Cost
#    - Role_Ambiguity      ← Switching_Cost
#    - Work_Home_Conflict  ← Work_Overload
#    - Willingness         ← Switching_Benefit
#    - Long_Term_Adoption  ← Willingness
#
#  Outcome:
#    - General_Grumbling   ← All others

def run_regression(y_name, x_names, data):
    """OLS regression with standardized path coefficients, p-values, R²."""
    Y = data[y_name].values
    X_raw = data[x_names].values
    n, k = X_raw.shape

    # Add intercept
    X = np.column_stack([np.ones(n), X_raw])

    # OLS: β = (X'X)^-1 X'y
    beta = np.linalg.lstsq(X, Y, rcond=None)[0]
    y_hat = X @ beta
    residuals = Y - y_hat

    # R²
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((Y - Y.mean())**2)
    r2 = 1 - ss_res / ss_tot
    r2_adj = 1 - (1 - r2) * (n - 1) / (n - k - 1)

    # Standard errors
    mse = ss_res / (n - k - 1)
    cov_beta = mse * np.linalg.inv(X.T @ X)
    se = np.sqrt(np.diag(cov_beta))

    # t-stats and p-values
    t_stats = beta / se
    p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n - k - 1))

    results = []
    for i, name in enumerate(x_names):
        results.append({
            'Path':        f"{name} → {y_name}",
            'From':        name,
            'To':          y_name,
            'Beta':        round(beta[i + 1], 4),
            'SE':          round(se[i + 1], 4),
            't_stat':      round(t_stats[i + 1], 3),
            'p_value':     round(p_values[i + 1], 4),
            'Significant': '***' if p_values[i+1] < 0.001 else
                           '**'  if p_values[i+1] < 0.01  else
                           '*'   if p_values[i+1] < 0.05  else
                           '(ns)'
        })
    return results, round(r2, 4), round(r2_adj, 4)


# ─────────────────────────────────────────────
# 4. ESTIMATE ALL STRUCTURAL EQUATIONS
# ─────────────────────────────────────────────
all_paths = []
r2_table  = []

# Equation 1: Work Overload ← Switching Cost
res, r2, r2a = run_regression('Work_Overload',
    ['Switching_Cost'], data_std)
all_paths += res
r2_table.append(('Work_Overload', r2, r2a))

# Equation 2: Role Ambiguity ← Switching Cost
res, r2, r2a = run_regression('Role_Ambiguity',
    ['Switching_Cost'], data_std)
all_paths += res
r2_table.append(('Role_Ambiguity', r2, r2a))

# Equation 3: Work-Home Conflict ← Work Overload
res, r2, r2a = run_regression('Work_Home_Conflict',
    ['Work_Overload'], data_std)
all_paths += res
r2_table.append(('Work_Home_Conflict', r2, r2a))

# Equation 4: Willingness ← Switching Benefit
res, r2, r2a = run_regression('Willingness',
    ['Switching_Benefit'], data_std)
all_paths += res
r2_table.append(('Willingness', r2, r2a))

# Equation 5: Long-Term Adoption ← Willingness
res, r2, r2a = run_regression('Long_Term_Adoption',
    ['Willingness'], data_std)
all_paths += res
r2_table.append(('Long_Term_Adoption', r2, r2a))

# Equation 6: General Grumbling ← ALL predictors (main outcome)
grumbling_predictors = [
    'Switching_Cost',
    'Switching_Benefit',
    'Work_Overload',
    'Role_Ambiguity',
    'Work_Home_Conflict',
    'Willingness',
    'Long_Term_Adoption'
]
res, r2, r2a = run_regression('General_Grumbling',
    grumbling_predictors, data_std)
all_paths += res
r2_table.append(('General_Grumbling', r2, r2a))


# ─────────────────────────────────────────────
# 5. COMPUTE INDIRECT & TOTAL EFFECTS ON GRUMBLING
# ─────────────────────────────────────────────
# Extract direct path betas into a dict for easy lookup
path_dict = {(p['From'], p['To']): p['Beta'] for p in all_paths}

def get_beta(frm, to):
    return path_dict.get((frm, to), 0.0)

# Indirect effects on General_Grumbling
indirect = {
    'Switching_Cost (via Work_Overload)':
        get_beta('Switching_Cost', 'Work_Overload') *
        get_beta('Work_Overload',  'General_Grumbling'),

    'Switching_Cost (via Role_Ambiguity)':
        get_beta('Switching_Cost',  'Role_Ambiguity') *
        get_beta('Role_Ambiguity',  'General_Grumbling'),

    'Switching_Cost (via Work_Overload → Work_Home_Conflict)':
        get_beta('Switching_Cost',      'Work_Overload') *
        get_beta('Work_Overload',       'Work_Home_Conflict') *
        get_beta('Work_Home_Conflict',  'General_Grumbling'),

    'Work_Overload (via Work_Home_Conflict)':
        get_beta('Work_Overload',      'Work_Home_Conflict') *
        get_beta('Work_Home_Conflict', 'General_Grumbling'),

    'Switching_Benefit (via Willingness)':
        get_beta('Switching_Benefit', 'Willingness') *
        get_beta('Willingness',       'General_Grumbling'),

    'Switching_Benefit (via Willingness → Long_Term_Adoption)':
        get_beta('Switching_Benefit',   'Willingness') *
        get_beta('Willingness',         'Long_Term_Adoption') *
        get_beta('Long_Term_Adoption',  'General_Grumbling'),

    'Willingness (via Long_Term_Adoption)':
        get_beta('Willingness',        'Long_Term_Adoption') *
        get_beta('Long_Term_Adoption', 'General_Grumbling'),
}

# Total effects = direct + all indirect
direct_on_grumbling = {
    c: get_beta(c, 'General_Grumbling') for c in grumbling_predictors
}

total_effects = {}
for c in grumbling_predictors:
    direct = direct_on_grumbling[c]
    indir  = sum(v for k, v in indirect.items() if k.startswith(c))
    total_effects[c] = round(direct + indir, 4)


# ─────────────────────────────────────────────
# 6. MODEL FIT (approximate via residual analysis)
# ─────────────────────────────────────────────
# Compute SRMR (Standardized Root Mean Square Residual)
# SRMR < 0.08 = good fit
observed_corr  = data_std.corr().values
n_vars         = len(data_std.columns)
var_names      = list(data_std.columns)

# Implied correlations via reproduced covariance from path coefficients
# Simple approximation: use R² values to gauge fit
grumbling_r2 = [r for name, r, ra in r2_table if name == 'General_Grumbling'][0]

# Compute residual correlations for SRMR
residuals_matrix = []
for i in range(n_vars):
    for j in range(i+1, n_vars):
        observed_r = data_std[var_names[i]].corr(data_std[var_names[j]])
        implied_r  = path_dict.get((var_names[i], var_names[j]),
                     path_dict.get((var_names[j], var_names[i]), 0.0))
        residuals_matrix.append((observed_r - implied_r)**2)

srmr = np.sqrt(np.mean(residuals_matrix))


# ─────────────────────────────────────────────
# 7. PRINT ALL RESULTS
# ─────────────────────────────────────────────

print("\n" + "="*70)
print("        STRUCTURAL EQUATION MODEL (SEM) RESULTS")
print("="*70)

# --- Direct Paths ---
print("\n" + "─"*70)
print("  SECTION 1: ALL DIRECT PATH COEFFICIENTS")
print("─"*70)
print(f"  {'Path':<52} {'Beta':>7} {'p-value':>9} {'Sig':>5}")
print("  " + "-"*65)

sections = [
    ("Paths TO Work Overload:",        'Work_Overload'),
    ("Paths TO Role Ambiguity:",       'Role_Ambiguity'),
    ("Paths TO Work-Home Conflict:",   'Work_Home_Conflict'),
    ("Paths TO Willingness:",          'Willingness'),
    ("Paths TO Long-Term Adoption:",   'Long_Term_Adoption'),
    ("Paths TO General Grumbling:",    'General_Grumbling'),
]

for section_label, target in sections:
    section_paths = [p for p in all_paths if p['To'] == target]
    if section_paths:
        print(f"\n  {section_label}")
        for p in section_paths:
            from_short = p['From'].replace('_', ' ')
            to_short   = p['To'].replace('_', ' ')
            path_str   = f"  {from_short} → {to_short}"
            print(f"  {path_str:<52} {p['Beta']:>7.4f} {p['p_value']:>9.4f} {p['Significant']:>5}")

# --- R² Table ---
print("\n\n" + "─"*70)
print("  SECTION 2: VARIANCE EXPLAINED (R²)")
print("─"*70)
print(f"  {'Construct':<25} {'R²':>8} {'Adj. R²':>10}  Interpretation")
print("  " + "-"*60)
for name, r2, r2a in r2_table:
    if r2 >= 0.26:
        interp = "Large effect"
    elif r2 >= 0.13:
        interp = "Medium effect"
    elif r2 >= 0.02:
        interp = "Small effect"
    else:
        interp = "Negligible"
    print(f"  {name.replace('_',' '):<25} {r2:>8.4f} {r2a:>10.4f}  {interp}")

# --- Indirect Effects ---
print("\n\n" + "─"*70)
print("  SECTION 3: INDIRECT EFFECTS ON GENERAL GRUMBLING")
print("─"*70)
print("  (These are effects that work THROUGH other constructs)")
print(f"\n  {'Indirect Path':<55} {'Effect':>8}")
print("  " + "-"*65)
for path_name, effect in sorted(indirect.items(), key=lambda x: abs(x[1]), reverse=True):
    flag = "✅" if abs(effect) > 0.02 else "  "
    print(f"  {flag} {path_name:<53} {effect:>8.4f}")

# --- Total Effects Ranking ---
print("\n\n" + "─"*70)
print("  SECTION 4: TOTAL EFFECTS ON GENERAL GRUMBLING (RANKING)")
print("─"*70)
print("  (Direct + Indirect — this is the TRUE impact of each construct)")
print(f"\n  {'Rank':<6} {'Construct':<25} {'Direct':>8} {'Indirect':>10} {'TOTAL':>8} {'Direction'}")
print("  " + "-"*65)

ranked = sorted(total_effects.items(), key=lambda x: abs(x[1]), reverse=True)
for rank, (construct, total) in enumerate(ranked, 1):
    direct  = direct_on_grumbling.get(construct, 0)
    indir   = sum(v for k, v in indirect.items() if k.startswith(construct))
    direction = "➕ Increases Grumbling" if total > 0 else "➖ Reduces Grumbling"
    print(f"  {rank:<6} {construct.replace('_',' '):<25} {direct:>8.4f} {indir:>10.4f} {total:>8.4f}  {direction}")

# --- Model Fit ---
print("\n\n" + "─"*70)
print("  SECTION 5: MODEL FIT SUMMARY")
print("─"*70)
print(f"""
  R² for General Grumbling : {grumbling_r2:.4f}
    → The model explains {grumbling_r2*100:.1f}% of variance in Grumbling

  SRMR (approx.)           : {srmr:.4f}
    → Rule: < 0.08 = good fit | < 0.05 = excellent fit

  Note: For full fit indices (CFI, TLI, RMSEA), use the semopy
  library results below (requires: pip install semopy)
""")

# ─────────────────────────────────────────────
# 8. SEMOPY (full SEM with fit indices)
# ─────────────────────────────────────────────
print("─"*70)
print("  SECTION 6: FULL SEM WITH SEMOPY (fit indices: CFI, RMSEA, SRMR)")
print("─"*70)

try:
    import semopy

    # semopy model specification
    # Each line = one structural equation
    # ~ means "is predicted by"
    model_spec = """
    # Mediator equations
    Work_Overload      ~ Switching_Cost
    Role_Ambiguity     ~ Switching_Cost
    Work_Home_Conflict ~ Work_Overload
    Willingness        ~ Switching_Benefit
    Long_Term_Adoption ~ Willingness

    # Outcome equation
    General_Grumbling ~ Switching_Cost + Switching_Benefit + Work_Overload + Role_Ambiguity + Work_Home_Conflict + Willingness + Long_Term_Adoption
    """

    model = semopy.Model(model_spec)
    result = model.fit(data_std)

    print("\n  --- semopy Path Estimates ---")
    estimates = model.inspect()
    print(estimates.to_string(index=False))

    print("\n  --- semopy Model Fit Indices ---")
    fit = semopy.calc_stats(model)
    print(fit.T.to_string())

except ImportError:
    print("""
  semopy not installed. To get full fit indices (CFI, TLI, RMSEA):
    pip install semopy
  Then re-run this script.

  The path analysis results in Sections 1-5 above are fully valid
  and interpretable without semopy.
""")

# ─────────────────────────────────────────────
# 9. FINAL INTERPRETATION GUIDE
# ─────────────────────────────────────────────
print("\n\n" + "="*70)
print("  FINAL ANSWER: WHAT MAKES STEM STUDENTS GRUMBLE THE MOST?")
print("="*70)
print("""
  Read Section 4 (Total Effects Ranking) for the definitive answer.

  HOW TO INTERPRET:
  ─────────────────
  Beta (path coefficient):
    • 0.10–0.19  = Small effect
    • 0.20–0.29  = Moderate effect
    • 0.30+      = Large/Strong effect
    • Negative   = This construct REDUCES grumbling

  Significance:
    • ***  p < 0.001  = Very highly significant
    • **   p < 0.01   = Highly significant
    • *    p < 0.05   = Significant
    • (ns) p > 0.05   = Not significant (may not be a real effect)

  Indirect Effects:
    • A construct can affect grumbling THROUGH another construct
    • e.g., Switching Cost → Work Overload → Grumbling
    • This is called MEDIATION

  R² for General Grumbling:
    • Tells you how much of grumbling your model explains
    • e.g., R²=0.45 means 45% of grumbling is explained by your constructs
    • The remaining % is due to factors not in the model

  ──────────────────────────────────────────────────────────────────────
  ✅  Share this output and we can interpret findings together,
      write up the results, or create visualizations of the path diagram.
  ──────────────────────────────────────────────────────────────────────
""")

✅ Valid responses: 212

        STRUCTURAL EQUATION MODEL (SEM) RESULTS

──────────────────────────────────────────────────────────────────────
  SECTION 1: ALL DIRECT PATH COEFFICIENTS
──────────────────────────────────────────────────────────────────────
  Path                                                    Beta   p-value   Sig
  -----------------------------------------------------------------

  Paths TO Work Overload:
    Switching Cost → Work Overload                      0.4355    0.0000   ***

  Paths TO Role Ambiguity:
    Switching Cost → Role Ambiguity                     0.3226    0.0000   ***

  Paths TO Work-Home Conflict:
    Work Overload → Work Home Conflict                  0.4909    0.0000   ***

  Paths TO Willingness:
    Switching Benefit → Willingness                     0.3325    0.0000   ***

  Paths TO Long-Term Adoption:
    Willingness → Long Term Adoption                    0.4557    0.0000   ***

  Paths TO General Grumbling:
    Switching Cost → Gener

In [1]:
"""
SEM Path Diagram — Grumbling Behavior in STEM Students
Pastel color scheme, 300 DPI, publication-ready
"""

import graphviz

# ── Pastel color palette ──────────────────────────────────────────────
C_ROOT      = "#FFB3B3"   # soft rose       — root causes (exogenous)
C_ROOT_BD   = "#E07070"
C_MED       = "#FFD9A0"   # warm peach      — mediators
C_MED_BD    = "#D4913A"
C_PROTECT   = "#B3D9FF"   # sky blue        — protective / reduces grumbling
C_PROTECT_BD= "#4A90C4"
C_OUTCOME   = "#C8F5C8"   # mint green      — outcome
C_OUTCOME_BD= "#3D9E3D"
C_FONT      = "#2D2D2D"

# ── Edge colors ───────────────────────────────────────────────────────
POS_SIG     = "#D94F4F"   # red             — positive significant path
POS_NS      = "#E8A0A0"   # light red       — positive non-significant
NEG_SIG     = "#3A7FC1"   # blue            — negative significant path
NEG_NS      = "#A0C4E8"   # light blue      — negative non-significant

dot = graphviz.Digraph(
    name="SEM_Grumbling",
    format="png",
    engine="dot",
)

dot.attr(
    rankdir="LR",
    bgcolor="white",
    fontname="Helvetica Neue",
    fontsize="13",
    dpi="300",
    size="16,10",          # inches → wide canvas
    ratio="fill",
    pad="0.6",
    nodesep="0.55",
    ranksep="1.6",
    splines="curved",
)

# ── Node defaults ─────────────────────────────────────────────────────
node_base = dict(
    shape="roundedbox",
    style="filled,setlinewidth(1.8)",
    fontname="Helvetica Neue",
    fontsize="13",
    fontcolor=C_FONT,
    margin="0.18,0.12",
    width="2.2",
    fixedsize="false",
)

def add_node(name, label, fill, border, bold=False):
    ff = "Helvetica Neue Bold" if bold else "Helvetica Neue"
    dot.node(name, label=label,
             fillcolor=fill, color=border,
             fontname=ff, **{k:v for k,v in node_base.items()
                             if k not in ("fontname",)})

# ── Nodes ─────────────────────────────────────────────────────────────

# Exogenous (root causes)
add_node("SC",  "Switching Cost\n(β_total = +0.295 ②)",      C_ROOT,    C_ROOT_BD,    bold=True)
add_node("SB",  "Switching Benefit\n(β_total = −0.175 ⑥)",   C_PROTECT, C_PROTECT_BD, bold=False)

# Mediators
add_node("WO",  "Work Overload\n(β_total = +0.177 ⑤)",       C_MED,     C_MED_BD)
add_node("RA",  "Role Ambiguity\n(β_total = +0.320 ①)",       C_ROOT,    C_ROOT_BD,    bold=True)
add_node("WHC", "Work-Home Conflict\n(β_total = +0.190 ③)",   C_MED,     C_MED_BD)
add_node("WI",  "Willingness\n(β_total = −0.183 ④)",          C_PROTECT, C_PROTECT_BD)
add_node("LTA", "Long-Term Adoption\n(β_total = −0.082 ⑦)",   C_PROTECT, C_PROTECT_BD)

# Outcome
add_node("GG",  "General\nGrumbling\n(R² = 0.43)",           C_OUTCOME, C_OUTCOME_BD, bold=True)

# ── Layout ranks ──────────────────────────────────────────────────────
with dot.subgraph() as s:
    s.attr(rank="same")
    s.node("SC"); s.node("SB")

with dot.subgraph() as s:
    s.attr(rank="same")
    s.node("WO"); s.node("RA"); s.node("WHC")
    s.node("WI"); s.node("LTA")

with dot.subgraph() as s:
    s.attr(rank="same")
    s.node("GG")

# ── Edge helper ───────────────────────────────────────────────────────
def edge(src, dst, beta, sig, curved=False, lp=None):
    """
    sig: '***','**','*','(ns)'
    beta: float
    """
    positive = beta >= 0
    significant = sig != "(ns)"

    color = (POS_SIG if (positive and significant) else
             POS_NS  if (positive and not significant) else
             NEG_SIG if (not positive and significant) else
             NEG_NS)

    style = "solid" if significant else "dashed"
    width = "2.4"   if significant else "1.4"
    arrow = "normal"

    sign  = "+" if beta >= 0 else ""
    label = f" {sign}{beta:.3f}{sig} "

    attrs = dict(
        label=label,
        color=color,
        fontcolor=color,
        fontname="Helvetica Neue",
        fontsize="11",
        penwidth=width,
        style=style,
        arrowsize="0.8",
        arrowhead=arrow,
    )
    if lp:
        attrs["lp"] = lp

    dot.edge(src, dst, **attrs)

# ── Structural paths ──────────────────────────────────────────────────

# Exogenous → Mediators
edge("SC",  "WO",  0.436, "***")
edge("SC",  "RA",  0.323, "***")
edge("WO",  "WHC", 0.491, "***")
edge("SB",  "WI",  0.333, "***")
edge("WI",  "LTA", 0.456, "***")

# → Grumbling (direct paths)
edge("RA",  "GG",  0.320,  "***")
edge("WHC", "GG",  0.190,  "**")
edge("WI",  "GG", -0.145,  "*")
edge("SC",  "GG",  0.115,  "(ns)")
edge("SB",  "GG", -0.114,  "(ns)")
edge("WO",  "GG",  0.083,  "(ns)")
edge("LTA", "GG", -0.082,  "(ns)")

# ── Legend ────────────────────────────────────────────────────────────
legend = """<
<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="4" CELLPADDING="6"
       BGCOLOR="white" COLOR="#CCCCCC">
  <TR><TD COLSPAN="2" ALIGN="CENTER"><B>Legend</B></TD></TR>
  <TR>
    <TD BGCOLOR="#FFB3B3" WIDTH="30"> </TD>
    <TD ALIGN="LEFT">Root Cause / Increases Grumbling</TD>
  </TR>
  <TR>
    <TD BGCOLOR="#FFD9A0"> </TD>
    <TD ALIGN="LEFT">Mediator</TD>
  </TR>
  <TR>
    <TD BGCOLOR="#B3D9FF"> </TD>
    <TD ALIGN="LEFT">Protective / Reduces Grumbling</TD>
  </TR>
  <TR>
    <TD BGCOLOR="#C8F5C8"> </TD>
    <TD ALIGN="LEFT">Outcome (General Grumbling)</TD>
  </TR>
  <TR><TD COLSPAN="2"> </TD></TR>
  <TR><TD COLSPAN="2" ALIGN="LEFT"><FONT COLOR="#D94F4F">─── Positive significant path</FONT></TD></TR>
  <TR><TD COLSPAN="2" ALIGN="LEFT"><FONT COLOR="#3A7FC1">─── Negative significant path</FONT></TD></TR>
  <TR><TD COLSPAN="2" ALIGN="LEFT"><FONT COLOR="#AAAAAA">- - Non-significant path</FONT></TD></TR>
  <TR><TD COLSPAN="2" ALIGN="LEFT">*** p&lt;.001  ** p&lt;.01  * p&lt;.05</TD></TR>
  <TR><TD COLSPAN="2" ALIGN="LEFT">β = Standardized path coefficient</TD></TR>
  <TR><TD COLSPAN="2" ALIGN="LEFT">① ② … = Total-effect rank</TD></TR>
</TABLE>>"""

dot.node("legend", label=legend,
         shape="plaintext", fontname="Helvetica Neue", fontsize="11",
         pos="0,0!")

# ── Title ─────────────────────────────────────────────────────────────
dot.attr(label=(
    "Grumbling Behavior in STEM Students Toward Non-STEM Courses\n"
),
    labelloc="t", labeljust="c",
    fontname="Helvetica Neue Bold", fontsize="15",
    fontcolor="#333333",
)

# ── Render ────────────────────────────────────────────────────────────
out = dot.render("./SEM_Grumbling_Diagram", cleanup=True)
print(f"Saved: {out}")


(process:21988): Pango-WARNING **: 15:41:28.179: couldn't load font "Helvetica Neue Bold Not-Rotated 15", falling back to "Sans Bold Not-Rotated 15", expect ugly output.

(process:21988): Pango-WARNING **: 15:41:28.205: couldn't load font "Helvetica Neue Bold Not-Rotated 13", falling back to "Sans Bold Not-Rotated 13", expect ugly output.

(process:21988): Pango-WARNING **: 15:41:28.213: couldn't load font "Helvetica Neue Not-Rotated 13", falling back to "Sans Not-Rotated 13", expect ugly output.

(process:21988): Pango-WARNING **: 15:41:28.229: couldn't load font "Helvetica Neue Bold Not-Rotated 11", falling back to "Sans Bold Not-Rotated 11", expect ugly output.

(process:21988): Pango-WARNING **: 15:41:28.231: couldn't load font "Helvetica Neue Not-Rotated 11", falling back to "Sans Not-Rotated 11", expect ugly output.


Saved: SEM_Grumbling_Diagram.png
